# Linux File Permissions (Educational Notebook)
This notebook explains the Linux permission model with examples and exercises.

## 1. Inodes and Metadata

Every file (including directories) has an **inode** storing metadata.

Important inode fields:
- **UID (User ID):** numeric owner identifier.
- **GID (Group ID):** numeric group identifier.
- **Permission bits (12 bits total):**
  - 9 permission bits: owner/group/others (rwx rwx rwx)
  - 3 special bits: setuid, setgid, sticky bit

> Linux often displays only the 9 regular permission bits unless the special bits are set.


## 2. Root User

The **root** user (UID 0) bypasses normal file permission checks in almost all situations. Root can usually read, write, execute, change ownership, and change permissions regardless of the permission bits.

## 3. Permission Classes

Every access attempt falls into one category:
- **Root** (UID 0): privileged superuser.
- **Owner (User):** file owner.
- **Group:** users belonging to the file's group.
- **Others (World):** everyone else.

## 4-6. Permission Bits and RWX

The 9 permission bits are:

| Owner | Group | Others |
|---|---|---|
| rwx | rwx | rwx |

Meaning:
- **r (4)** = read
- **w (2)** = write
- **x (1)** = execute/search

**Full permission = rwx = 7 (4+2+1).**


## 7-9. ls -l and stat

Example:
```bash
$ ls -l hello.sh
-rwxr-xr-- 1 ali staff 177 Jul 5 23:28 hello.sh
```

Fields:
1. File type + permissions
2. Hard link count
3. Owner
4. Group
5. Size
6. Date/time
7. Filename

`stat hello.sh` shows much more information:
- inode number
- size
- owner/group IDs
- permissions (symbolic and octal)
- timestamps
- device information


## 10. Numeric Permissions

Mapping:

|Number|Meaning|
|---:|---|
|0|---|
|1|--x|
|2|-w-|
|3|-wx|
|4|r--|
|5|r-x|
|6|rw-|
|7|rwx|

Example:
- **0755** = owner=rwx, group=r-x, others=r-x.
The leading **0** denotes octal notation.


## 11. Concrete Example

`report.txt` with permission `0644`:
- Owner: read/write
- Group: read
- Others: read

## 12-19. Meaning on Files vs Directories

|Permission|File|Directory|
|---|---|---|
|r|Read file contents (`cat`, `less`, `cp`)|List directory names (`ls`)|
|w|Modify file|Create/delete/rename entries (usually with x too)|
|x|Run program/script|Enter/search directory (`cd`) and access files by name|

Examples:
- File with `r--`: `cat`, `cp` work; editing does not.
- File with `--x`: executable but not readable (rare).
- Directory with `r-x`: can list and enter.
- Directory with `--x`: cannot list names, but can access known filenames.
- Directory with `wx`: create/delete entries if names are known.

Deleting, renaming, or moving a file depends mainly on the **directory's** permissions, not the file's permissions.
A writable directory (`w+x`) can allow deletion even if the file itself is read-only.

Granting `w` on a shared directory can be dangerous because users may delete or rename files inside it.


## 18. Permissions for Programs

Programs like `ls`, `cp`, and `cat` are executable files (typically in `/bin` or `/usr/bin`).
Users normally have **r-x** access, allowing execution.

## 20-24. chmod, chown, chgrp

**chmod** changes permissions.
- Owner may change permissions.
- Root may always change permissions.

Examples:
```bash
chmod 644 file.txt
chmod u+x script.sh
chmod -R 755 project/
```

**chown** changes owner (normally only root).
```bash
sudo chown alice file.txt
sudo chown alice:developers file.txt
```

**chgrp** changes group.
- Root can always do it.
- File owner may change the group to one they belong to (Linux systems commonly allow this).

```bash
chgrp developers file.txt
chgrp -R developers project/
```

`-R` means recursive for directories.


## 25-29. Practical Notes

`chmod 000 important.txt`
- Nobody except root can access it.
- Useful to temporarily protect important files.

`chmod 200 file`
- Owner has write only.
- Can overwrite/truncate but cannot read contents.

`chmod 100 file`
- Owner execute only.
- Useful only for executable binaries/scripts.

`chmod 100 directory`
- Can enter directory if name known, but cannot list or modify.

For directories, **500 (r-x)** is often a practical minimum because users can enter and read names without modifying contents.


## 30-32. Ownership and 777

Change owner/group:
```bash
sudo chown ali file
sudo chown ali:students file
sudo chgrp students file
```

**777 = rwxrwxrwx**
- Everyone can read, write, execute.
- Usually **unsafe**.
- Avoid except for special temporary situations.


## Worked Example

```
-rwxrwxrwx 1 ali ali    27 Jul  5 23:31 a.txt
-rw-rw-r-- 1 ali ali    36 Jul  5 23:31 b.txt
-rw-rw-r-- 1 ali ali     0 Jul  6 02:12 bar
-rwxrwxr-x 1 ali ali   177 Jul  5 23:28 count_words.sh
-rw-rw-r-- 1 ali ali     0 Jul  5 22:39 fee
drwxrwxr-x 2 ali ali  4096 Jul  6 00:08 foo
-rwxrwxr-x 2 ali ali 70448 Jul  6 01:03 hello
-rw-rw-r-- 1 ali ali    64 Jul  6 01:03 hello.c
```

- `a.txt`: world writable (777) → unsafe.
- `b.txt`: 664.
- `count_words.sh`: executable script (775).
- `foo`: directory (775).
- `hello`: executable program.
- `hello.c`: C source file.


## Review Questions

1. Why is deleting a file controlled by the directory permission?
2. Difference between `644` and `755`?
3. Why is `777` dangerous?
4. What does `chmod 200 file` do?
5. Why can root usually ignore permission bits?
6. When would `chmod 000` be useful?
7. What does `-R` do?
8. Difference between `chown` and `chgrp`?


In [ ]:
# Try these commands on your Linux system
touch demo.txt
mkdir demo_dir
ls -l
stat demo.txt
chmod 644 demo.txt
chmod 755 demo_dir
ls -ld demo_dir
